# THYROID_2026 — Statistical Analysis Workbench

**Notebook 05** — Worked examples for the `ThyroidStatisticalAnalyzer` module.

Demonstrates:
1. Connection setup (MotherDuck or local DuckDB)
2. Table 1 generation with automatic test selection
3. Hypothesis testing with FDR correction
4. Logistic regression (BRAF/ETE → recurrence) with VIF + OR forest plot
5. Cox proportional hazards model (time to event) with Schoenfeld check
6. Correlation matrix with significance stars
7. Missing data audit
8. Publication export bundle

> **PHI note:** This notebook operates on `research_id`-keyed data only. No note text is displayed.

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
import numpy as np
import pandas as pd
import plotly.io as pio

pio.renderers.default = "notebook"
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

print("Environment ready.")

## 1. Connect to Data

Set `USE_LOCAL = True` to use the local DuckDB fallback (`thyroid_master_local.duckdb`).
Set `USE_LOCAL = False` and export `MOTHERDUCK_TOKEN` to connect to MotherDuck.

In [ ]:
import duckdb
from motherduck_client import MotherDuckClient, MotherDuckConfig

USE_LOCAL = os.getenv("USE_LOCAL_DUCKDB", "false").lower() in ("1", "true", "yes")

if USE_LOCAL:
    con = duckdb.connect("thyroid_master_local.duckdb")
    print("Connected to local DuckDB")
else:
    # Load token from .streamlit/secrets.toml if not in environment
    if not os.getenv("MOTHERDUCK_TOKEN"):
        try:
            import toml
            secrets = toml.load(".streamlit/secrets.toml")
            os.environ["MOTHERDUCK_TOKEN"] = secrets["MOTHERDUCK_TOKEN"]
        except Exception:
            raise RuntimeError("Set MOTHERDUCK_TOKEN or add it to .streamlit/secrets.toml")
    client = MotherDuckClient(MotherDuckConfig(database="thyroid_research_2026"))
    con = client.connect_rw()
    print("Connected to MotherDuck")

In [ ]:
from utils.statistical_analysis import (
    ThyroidStatisticalAnalyzer,
    THYROID_TABLE1_PRESET,
    THYROID_OUTCOMES,
    THYROID_PREDICTORS,
    THYROID_SURVIVAL,
)

analyzer = ThyroidStatisticalAnalyzer(con)

# Resolve the best available analytic view
source_view = analyzer.resolve_view("risk_enriched_mv")
print(f"Using view: {source_view}")

## 2. Load Cohort Data

In [ ]:
# Load a representative analytic cohort
df = con.execute(f"SELECT * FROM {source_view}").fetchdf()
print(f"Cohort: {len(df):,} rows × {len(df.columns)} columns")
df.head(3)

## 3. Missing Data Audit

In [ ]:
miss = analyzer.missing_data_summary(df)
miss_nonzero = miss[miss["pct_missing"] > 0].head(20)
print(f"Columns with any missing data: {len(miss[miss['pct_missing'] > 0])}")
miss_nonzero

## 4. Table 1 — Cohort Characteristics

Generates a publication-ready Table 1. Variables are automatically classified as
continuous (median [IQR]) or categorical (n, %). Non-normal distributions are
detected via Shapiro-Wilk and reported with appropriate non-parametric tests.

When grouped, p-values use:
- **Continuous**: Welch t-test (normal) or Mann-Whitney U (non-normal); ANOVA or Kruskal-Wallis for 3+ groups
- **Categorical**: Chi-square or Fisher exact (expected cell < 5)

In [ ]:
# Overall Table 1 (no grouping)
t1_overall, meta_overall = analyzer.generate_table_one(
    data=df,
    continuous_vars=THYROID_TABLE1_PRESET["continuous"],
    categorical_vars=THYROID_TABLE1_PRESET["categorical"],
)
print(f"N = {meta_overall['n_total']:,}")
t1_overall

In [ ]:
# Grouped by BRAF status (if available)
groupby_col = "braf_positive" if "braf_positive" in df.columns else None

if groupby_col:
    t1_grouped, meta_grouped = analyzer.generate_table_one(
        data=df,
        groupby_col=groupby_col,
        continuous_vars=THYROID_TABLE1_PRESET["continuous"],
        categorical_vars=THYROID_TABLE1_PRESET["categorical"],
    )
    print(f"Non-normal continuous vars: {meta_grouped.get('nonnormal', [])}")
    t1_grouped
else:
    print("braf_positive not available — skipping grouped Table 1")

## 5. Hypothesis Testing with Multiple Comparison Correction

Auto-selects test type per variable. FDR (Benjamini-Hochberg) correction applied
across all features. Effect sizes: Cohen's d (continuous), Cramér's V (categorical).

In [ ]:
outcome_var = next((o for o in THYROID_OUTCOMES if o in df.columns), None)
features = [f for f in THYROID_PREDICTORS if f in df.columns and f != outcome_var]

if outcome_var and features:
    ht_results = analyzer.run_hypothesis_tests(
        data=df,
        target_var=outcome_var,
        feature_list=features,
        correction="fdr_bh",
    )
    n_sig = (ht_results["p_adjusted"] < 0.05).sum() if "p_adjusted" in ht_results.columns else "?"
    print(f"Features tested: {len(ht_results)} | Significant (FDR-adjusted): {n_sig}")
    ht_results
else:
    print(f"Outcome '{outcome_var}' or features not available in this view.")

## 6. Logistic Regression — BRAF/ETE Predicting Recurrence

Returns:
- **OR table** with 95% CI, z-statistic, p-value, significance flag, plain-English interpretation
- **VIF** for multicollinearity detection (VIF > 5 = moderate concern, > 10 = high)
- **AUC** via scikit-learn `roc_auc_score`
- **Pseudo-R²** (McFadden), AIC, BIC

In [ ]:
outcome = next((o for o in THYROID_OUTCOMES if o in df.columns), None)
predictors = [p for p in THYROID_PREDICTORS if p in df.columns and p != outcome]

if outcome and len(predictors) >= 2:
    lr_result = analyzer.fit_logistic_regression(
        outcome=outcome,
        predictors=predictors[:8],   # limit for demo
        data=df,
    )

    if "error" in lr_result:
        print(f"Error: {lr_result['error']}")
    else:
        print(f"N = {lr_result['n_obs']:,} | AUC = {lr_result.get('auc', 'N/A')} "
              f"| Pseudo-R² = {lr_result['pseudo_r2']} | AIC = {lr_result['aic']}")
        for w in lr_result.get("warnings", []):
            print(f"  ⚠️  {w}")
        print("\n--- Odds Ratio Table ---")
        display(lr_result["or_table"])
else:
    print("Insufficient variables for logistic regression demo.")

In [ ]:
# Forest plot of odds ratios
if "or_table" in locals() and not lr_result.get("or_table", pd.DataFrame()).empty:
    or_table = lr_result["or_table"]
    display_df = or_table[or_table["predictor"] != "const"].rename(columns={
        "predictor": "label", "OR": "estimate",
        "CI_lower": "ci_lower", "CI_upper": "ci_upper",
    })
    if not display_df.empty:
        fig_or = ThyroidStatisticalAnalyzer.create_forest_plot(
            display_df,
            title="Odds Ratios for Recurrence (Logistic Regression)",
            reference_value=1.0,
        )
        fig_or.show()

## 7. Cox Proportional Hazards Model

Uses `lifelines.CoxPHFitter` with penalizer=0.01. Reports:
- **HR table** with 95% CI, p-value, significance flag
- **Concordance index** (C-statistic)
- **Schoenfeld residuals** proportional hazards check
- Warnings for < 10 events (unreliable estimates)

In [ ]:
# Use survival cohort if available
surv_view = analyzer.resolve_view("survival_cohort_enriched")
if surv_view:
    df_surv = con.execute(f"SELECT * FROM {surv_view} LIMIT 5000").fetchdf()
    print(f"Survival cohort: {len(df_surv):,} rows")
    
    cox_preds = [p for p in THYROID_PREDICTORS if p in df_surv.columns]
    if len(cox_preds) >= 2:
        cox_result = analyzer.fit_cox_ph(
            time_col=THYROID_SURVIVAL["time_col"],
            event_col=THYROID_SURVIVAL["event_col"],
            predictors=cox_preds[:6],
            data=df_surv,
        )
        if "error" in cox_result:
            print(f"Error: {cox_result['error']}")
        else:
            print(f"N = {cox_result['n_obs']:,} | Events = {cox_result['n_events']} "
                  f"| Concordance = {cox_result['concordance']}")
            for w in cox_result.get("warnings", []):
                print(f"  ⚠️  {w}")
            display(cox_result["hr_table"])
    else:
        print("Insufficient predictors in survival cohort.")
else:
    print("survival_cohort_enriched not available. Run scripts/26_motherduck_materialize_v2.py --md")

In [ ]:
# Forest plot of hazard ratios
if "cox_result" in dir() and "hr_table" in cox_result and not cox_result["hr_table"].empty:
    hr_df = cox_result["hr_table"].rename(columns={
        "covariate": "label", "HR": "estimate",
        "CI_lower": "ci_lower", "CI_upper": "ci_upper",
    })
    fig_hr = ThyroidStatisticalAnalyzer.create_forest_plot(
        hr_df,
        title="Hazard Ratios — Cox PH Model",
        reference_value=1.0,
    )
    fig_hr.show()

## 8. Correlation Matrix

Spearman pairwise correlations with significance stars (*/\*\*/\*\*\*).
Uses pingouin when available for faster computation, falls back to scipy.

In [ ]:
numeric_cols = [
    c for c in THYROID_TABLE1_PRESET["continuous"] if c in df.columns
]

if len(numeric_cols) >= 2:
    corr, pval = analyzer.correlation_matrix_with_pvalues(
        df, numeric_cols, method="spearman"
    )
    print("Correlation matrix:")
    display(corr)
    
    fig_corr = ThyroidStatisticalAnalyzer.create_correlation_heatmap(
        corr, pval, title="Spearman Correlations (continuous variables)"
    )
    fig_corr.show()
else:
    print("Not enough numeric columns for correlation matrix.")

## 9. Export Publication Bundle

Save all results to `exports/statistical_workbench_YYYYMMDD/`.

In [ ]:
from datetime import datetime
from pathlib import Path
import json

EXPORT_DIR = Path(ROOT / "exports" / f"statistical_workbench_{datetime.now():%Y%m%d_%H%M}")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save Table 1
if "t1_overall" in dir() and not t1_overall.empty:
    t1_path = EXPORT_DIR / "table1_overall.csv"
    t1_overall.to_csv(t1_path)
    print(f"Table 1 → {t1_path}")

# Save logistic regression
if "lr_result" in dir() and "or_table" in lr_result:
    lr_path = EXPORT_DIR / "logistic_regression_or_table.csv"
    lr_result["or_table"].to_csv(lr_path, index=False)
    print(f"Logistic OR table → {lr_path}")

# Save Cox PH
if "cox_result" in dir() and "hr_table" in cox_result:
    cox_path = EXPORT_DIR / "cox_ph_hr_table.csv"
    cox_result["hr_table"].to_csv(cox_path, index=False)
    print(f"Cox HR table → {cox_path}")

# Save hypothesis testing
if "ht_results" in dir() and not ht_results.empty:
    ht_path = EXPORT_DIR / "hypothesis_tests.csv"
    ht_results.to_csv(ht_path, index=False)
    print(f"Hypothesis tests → {ht_path}")

# Save metadata manifest
manifest = {
    "generated": datetime.now().isoformat(),
    "source_view": source_view,
    "n_cohort": len(df),
    "random_seed": 42,
    "description": "Statistical Workbench output bundle — THYROID_2026",
}
(EXPORT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\nExport bundle: {EXPORT_DIR}")

## 10. Notes for Manuscript Integration

### Statistical reporting standards
- All p-values are two-sided unless noted
- Random seed = 42 for all sampling (Shapiro-Wilk subsampling)
- Logistic regression uses complete-case analysis (listwise deletion)
- Cox PH uses `penalizer=0.01` for numerical stability
- Effect sizes: Cohen's d (continuous), Cramér's V (categorical)

### View priority for analyses
1. `risk_enriched_mv` — broadest feature set, includes recurrence/risk flags
2. `survival_cohort_enriched` — time-to-event + 61k follow-up records (use for Cox)
3. `advanced_features_v3` — molecular flags, AJCC staging, ETE subtype
4. `ptc_cohort` — PTC-only subset

### Phase-2 extensions (planned)
- Competing risks (Fine-Gray) for cancer-specific vs. all-cause mortality
- Mixed-effects models for serial Tg/TSH trajectories
- SHAP explainability for ML models
- PSM refinements (caliper sensitivity, doubly-robust estimation)

See `studies/proposal2_ete_staging/` for the PSM results and `studies/manuscript_tables/` for LaTeX-ready tables.